# 03 — Evaluation

Compare three checkpoints side by side: base distilgpt2, the SFT model, and the DPO-aligned model. For each, we generate bios on 10 held-out prompts (not seen during training), then compute automatic metrics (average word count, generic phrase rate) and display qualitative examples. The goal is to verify that SFT improves domain fit and DPO improves preference alignment relative to the base.

In [ ]:
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import subprocess, sys
repo_root = "/content/aipi590-challenge-2"
token = os.environ["GITHUB_TOKEN"]
if not os.path.exists(repo_root):
    subprocess.run(
        f"git clone https://{token}@github.com/jonasneves/aipi590-challenge-2.git {repo_root}",
        shell=True, check=True
    )

!pip install -q trl peft transformers datasets accelerate bitsandbytes

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Always reload colab_utils so a re-run picks up the latest version from the repo
import importlib, src.colab_utils as _cu
importlib.reload(_cu)
from src.colab_utils import prepare_notebook, ensure_requirements, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")

# Reload again after prepare_notebook pulls the latest code
importlib.reload(_cu)
from src.colab_utils import prepare_notebook, ensure_requirements, publish_artifacts

ensure_requirements(repo_root)


In [ ]:
import json

pref_path = paths["data"] / "preferences.json"
records = json.loads(pref_path.read_text())

# Hold out the last 10 entries as eval prompts
held_out = records[-10:]
prompts = [r["prompt"] for r in held_out]

print(f"Loaded {len(prompts)} held-out prompts:")
for i, p in enumerate(prompts):
    print(f"  {i+1}. {p}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from src.eval import generate_bios, summarize, save_results

DRIVE_ROOT = "/content/drive/MyDrive/aipi590-challenge-2"

def resolve(name, local):
    """Return local path if it exists, else Drive path, else None."""
    if Path(local).exists():
        return local
    drive = f"{DRIVE_ROOT}/{name}"
    if Path(drive).exists():
        return drive
    return None

CHECKPOINTS = {
    "base": "distilgpt2",
    "sft":  resolve("sft_model",  "/content/sft_model"),
    "dpo":  resolve("dpo_model",  "/content/dpo_model"),
}

results = {}
for label, path in CHECKPOINTS.items():
    if path is None:
        print(f"SKIP {label}: not found in /content or Drive")
        continue
    try:
        bios = generate_bios(path, prompts)
        results[label] = {"bios": bios, "metrics": summarize(label, bios)}
        print(f"OK {label} ({path}): generated {len(bios)} bios")
    except Exception as e:
        print(f"SKIP {label}: {e}")


In [ ]:
# Side-by-side comparison for the first 3 prompts
available = list(results.keys())
n_examples = min(3, len(prompts))

for i in range(n_examples):
    print(f"\n{'='*70}")
    print(f"Prompt {i+1}: {prompts[i]}")
    print('='*70)
    for label in available:
        bio = results[label]["bios"][i] if i < len(results[label]["bios"]) else "(no output)"
        print(f"[{label.upper():8s}] {bio}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels   = list(results.keys())
avg_words = [results[l]["metrics"]["avg_words"]          for l in labels]
generic   = [results[l]["metrics"]["generic_rate"] for l in labels]

x = np.arange(len(labels))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(x, avg_words, width)
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel("Avg words")
ax1.set_title("Average Bio Length")

ax2.bar(x, generic, width, color="orange")
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_ylabel("Rate")
ax2.set_title("Generic Phrase Rate (lower is better)")

plt.tight_layout()
chart_path = paths["results"] / "eval_metrics.png"
plt.savefig(chart_path, dpi=150)
plt.show()

In [ ]:
all_metrics = [v["metrics"] for v in results.values()]
save_results(all_metrics, paths["results"] / "eval_metrics.json")

publish_artifacts(
    [paths["results"] / "eval_metrics.png", paths["results"] / "eval_metrics.json"],
    "add evaluation results",
    repo_root,
)